# 04. Regression: what predicts county Medicare spending?

**Model** (one row per county-year, 2014-2024):

spend_pc_std = b1·income + b2·poverty + b3·rural + b4·avg_age + b5·dual% + b6·MA% + year fixed effects + error

* Standard errors clustered by state (50 clusters).
* **Robustness check:** the same model weighted by number of beneficiaries (WLS),
  so a county with 100,000 beneficiaries counts more than one with 600.

Stata equivalents, for reference:

```stata
reg spend_pc_std income_10k poverty_rate rural avg_age dual_pct ma_rate i.year, vce(cluster state)
reg spend_pc_std income_10k poverty_rate rural avg_age dual_pct ma_rate i.year [aw=benes], vce(cluster state)
```

These are associations across counties, not causal effects.

In [ ]:
import duckdb
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

## 1. Data

Variables are rescaled in SQL so coefficients are easy to read:

| Variable | Unit |
|---|---|
| `income_10k` | median household income in $10,000s |
| `poverty_rate` | percentage points (already) |
| `rural` | 1 = nonmetro (RUCC 4-9), 0 = metro |
| `avg_age` | years |
| `dual_pct` | % of beneficiaries also on Medicaid, 0-100 |
| `ma_rate` | % of all Medicare enrollees in Medicare Advantage, 0-100 |

In [ ]:
with duckdb.connect("../medicare.duckdb", read_only=True) as con:
    df = con.sql("""
        SELECT year, fips, county_name, state, benes, spend_pc_std,
               median_income / 10000 AS income_10k,
               poverty_rate,
               CASE metro_rural WHEN 'Rural' THEN 1 WHEN 'Metro' THEN 0 END AS rural,
               avg_age,
               dual_pct * 100 AS dual_pct,
               ma_rate  * 100 AS ma_rate
        FROM merged
    """).df()

# statsmodels drops rows with missing values inside the regression, but the
# cluster groups must line up row for row, so drop them up front instead.
model_vars = ["spend_pc_std", "income_10k", "poverty_rate", "rural", "avg_age", "dual_pct", "ma_rate"]
reg = df.dropna(subset=model_vars).copy()
print(f"{len(df):,} rows, {len(reg):,} with complete data, {reg['state'].nunique()} states")

In [ ]:
reg[model_vars].describe().T.round(2)

## 2. Estimate

`C(year)` adds a dummy for each year (like `i.year` in Stata). It absorbs
anything that affects every county in a given year: national payment updates,
inflation, the 2020 COVID drop.

In [ ]:
formula = "spend_pc_std ~ income_10k + poverty_rate + rural + avg_age + dual_pct + ma_rate + C(year)"
clusters = {"groups": reg["state"]}

ols = smf.ols(formula, data=reg).fit(cov_type="cluster", cov_kwds=clusters)
wls = smf.wls(formula, data=reg, weights=reg["benes"]).fit(cov_type="cluster", cov_kwds=clusters)

main_vars = ["income_10k", "poverty_rate", "rural", "avg_age", "dual_pct", "ma_rate"]
print(summary_col(
    [ols, wls], model_names=["OLS", "WLS (weighted by benes)"], stars=True,
    regressor_order=main_vars, drop_omitted=True,
    info_dict={"N": lambda r: f"{int(r.nobs):,}", "Year FE": lambda r: "Yes"},
))

Side by side with 95% confidence intervals:

In [ ]:
def tidy(res, name):
    ci = res.conf_int().loc[main_vars]
    return pd.DataFrame({"model": name, "coef": res.params[main_vars], "se": res.bse[main_vars],
                         "ci_low": ci[0], "ci_high": ci[1], "p": res.pvalues[main_vars]})

coefs = pd.concat([tidy(ols, "OLS"), tidy(wls, "WLS")])
coefs.round(3)

## 3. Extra robustness: drop Miller County, GA

Miller County is the highest-spending county in 2023-2024 (about $30K per
beneficiary, driven by skilled nursing facility spending). Does one county move the results?

In [ ]:
no_miller = reg[reg["fips"] != "13201"]
ols_nm = smf.ols(formula, data=no_miller).fit(cov_type="cluster", cov_kwds={"groups": no_miller["state"]})
pd.DataFrame({"OLS": ols.params[main_vars], "OLS without Miller": ols_nm.params[main_vars]}).round(1)

## 4. Reading the results

*(Numbers below are from the run on 2026-11-02; rerun to confirm.)*

* **Rural:** after controls, rural counties spend about **$620 less** per
  beneficiary than metro counties (about $860 less when weighted). Significant at 1% in both.
* **Poverty:** each additional point of poverty is associated with about
  **$80-90 more** per beneficiary.
* **Income:** small and not significant once poverty is in the model. Income and
  poverty are correlated at about -0.74, so the model has a hard time separating them.
* **Age and dual eligibility:** both positive and significant. Dual-eligible
  beneficiaries tend to be sicker and lower income.
* **Medicare Advantage share:** close to zero and not significant.
* **Weighting** strengthens the rural gap, but signs and significance are the
  same, so the small counties are not driving the results.
* **Fit:** R² of about 0.42 (OLS) and 0.47 (WLS), including the year effects.

**Caveat:** standardized spending removes price differences, not health
differences. Some of what looks like "rural counties spend less" could be
differences in health status or in access to care.